# Validación de los tiempos en coche con el MITECO

`build_travel_matrix.ipynb` calcula tiempos en coche **de flujo libre** (sin tráfico, sin curvas, sin semáforos).
Este notebook los compara con un dato oficial independiente: el **tiempo de acceso al hospital** por concejo del
MITECO (Reto Demográfico, campo `tipo_hosp`, preparado en `03_data/miteco/explore_miteco_data.ipynb`).

**Qué sabemos y qué no del indicador del MITECO:**
- Hay un valor por concejo (78) y el nombre del hospital al que se refiere (`nom_hosp`).
- La unidad no está documentada; por el rango (de 1 a 54) se asume que son **minutos** (inferencia).
- No sabemos desde qué punto del concejo se mide (capital, centroide o media ponderada). Por eso se comparan dos
  versiones nuestras: desde la **capital** del concejo y la **media ponderada por población** de sus núcleos.
- Su hospital incluye privados y psiquiátricos (Centro Médico de Asturias, Sanatorio Adaro, Clínica Somió) y
  hospitales de fuera de Asturias (Sierrallana, Valle de Laciana). Nosotros usamos los 9 hospitales públicos de
  referencia.

**Diseño de la comparación.** Para que sea justa, se compara el tiempo **al mismo hospital**: solo en los concejos
en los que el hospital del MITECO es uno de nuestros 9. En el resto, las diferencias se deberían a la definición de
hospital y no a la red.

**Antes de ejecutar:** `build_travel_matrix.ipynb` ya ejecutado (capas `services` y `access` en `asturias.gpkg`).

In [ ]:
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

GPKG_PATH = os.path.join("data", "processed", "asturias.gpkg")
MITECO_PATH = os.path.join("..", "miteco", "out", "asturias_miteco.gpkg")

pd.set_option("display.width", 200)

## 1. Datos

- Núcleos con su concejo y población, y la tabla `access` filtrada a hospitales (los 5 más cercanos por núcleo).
- Indicador del MITECO por concejo: tiempo (`tipo_hosp`) y hospital (`nom_hosp`).

Los hospitales del MITECO se emparejan con los nuestros con una tabla escrita a mano (son pocos y así se ve cuál es
cuál).

In [ ]:
# MITECO hospital name -> id of the same hospital among our 9 reference hospitals (services layer).
# The rest of MITECO's hospitals are private, psychiatric or outside Asturias.
MITECO_TO_SERVICE_ID = {
    "HOSPITAL DE JARRIO": "hosp-104",
    "HOSPITAL DEL ORIENTE DE ASTURIAS FRANCISCO GRANDE COVIAN": "hosp-107",
    "HOSPITAL UNIVERSITARIO SAN AGUSTIN": "hosp-109",
    "HOSPITAL CARMEN Y SEVERO OCHOA": "hosp-102",
    "HOSPITAL VALLE DEL NALON": "hosp-111",
    "HOSPITAL VITAL ALVAREZ BUYLLA": "hosp-112",
}

settlements = gpd.read_file(GPKG_PATH, layer="settlements", ignore_geometry=True)
services = gpd.read_file(GPKG_PATH, layer="services", ignore_geometry=True)
access = gpd.read_file(GPKG_PATH, layer="access", ignore_geometry=True)
hospital_access = access[access["service_type"] == "hospital"].merge(
    settlements[["settlement_id", "municipality_code", "population", "type", "name"]], on="settlement_id")

hospital_names = services[services["service_type"] == "hospital"].set_index("service_id")["name"]
assert set(MITECO_TO_SERVICE_ID.values()) <= set(hospital_names.index), "unknown hospital id in the mapping"

miteco = gpd.read_file(MITECO_PATH, ignore_geometry=True)
miteco = miteco.rename(columns={"codmun_ine": "municipality_code", "nombre": "municipality",
                                "tipo_hosp": "miteco_minutes", "nom_hosp": "miteco_hospital"})
miteco = miteco[["municipality_code", "municipality", "miteco_minutes", "miteco_hospital"]]
miteco["service_id"] = miteco["miteco_hospital"].map(MITECO_TO_SERVICE_ID)

print("Hospital del MITECO -> nuestro hospital (vacío = no es uno de los 9 de referencia):")
counts = miteco.groupby("miteco_hospital")["service_id"].agg(["first", "size"]).sort_values("size", ascending=False)
counts["ours"] = counts["first"].map(hospital_names)
print(counts[["size", "ours"]].to_string())

## 2. Nuestros tiempos por concejo

Para cada concejo, dos medidas al hospital **más cercano** y dos al **hospital que dice el MITECO**:
- `capital`: desde el núcleo de tipo "Capitales de municipio";
- `weighted`: media de todos sus núcleos ponderada por población.

El hospital del MITECO puede no estar entre nuestros 5 más cercanos de algún núcleo; esos núcleos quedan sin
tiempo y se cuentan.

In [ ]:
def weighted_mean(group, column):
    weights = group["population"].clip(lower=0)
    return np.average(group[column], weights=weights) if weights.sum() > 0 else group[column].mean()


nearest = hospital_access[hospital_access["rank"] == 1]
ours = nearest.groupby("municipality_code").apply(lambda g: pd.Series({
    "nearest_capital": g.loc[g["type"] == "Capitales de municipio", "minutes"].mean(),
    "nearest_weighted": weighted_mean(g, "minutes"),
}), include_groups=False).reset_index()

to_miteco_hospital = hospital_access.merge(miteco[["municipality_code", "service_id"]],
                                           on=["municipality_code", "service_id"])
same = to_miteco_hospital.groupby("municipality_code").apply(lambda g: pd.Series({
    "same_capital": g.loc[g["type"] == "Capitales de municipio", "minutes"].mean(),
    "same_weighted": weighted_mean(g, "minutes"),
    "settlements_reached": len(g),
}), include_groups=False).reset_index()

comparison = (miteco.merge(ours, on="municipality_code", how="left")
              .merge(same, on="municipality_code", how="left")
              .merge(settlements.groupby("municipality_code").size().rename("settlements").reset_index(),
                     on="municipality_code", how="left"))
matched = comparison[comparison["service_id"].notna()]
print(f"Concejos: {len(comparison)}; con el mismo hospital que el MITECO: {len(matched)}")
print("Concejos emparejados en los que algún núcleo no tiene el hospital del MITECO entre sus 5 más cercanos:",
      int((matched["settlements_reached"] < matched["settlements"]).sum()))

## 3. Comparación

Para cada medida nuestra frente al MITECO:
- **cociente mediano** MITECO / nuestro: cuánto hay que multiplicar nuestros tiempos para llegar a los suyos;
- **error absoluto medio** (minutos) y **sesgo** medio (MITECO − nuestro);
- **correlación de Spearman**: si ordenamos los concejos igual (lo importante para preguntas del tipo "¿cuál está
  peor?").

In [ ]:
def compare(frame, column):
    pair = frame[["miteco_minutes", column]].dropna()
    diff = pair["miteco_minutes"] - pair[column]
    return pd.Series({
        "concejos": len(pair),
        "cociente mediano": (pair["miteco_minutes"] / pair[column]).median(),
        "error abs. medio (min)": diff.abs().mean(),
        "sesgo medio (min)": diff.mean(),
        "Spearman": pair["miteco_minutes"].corr(pair[column], method="spearman"),
    })


summary = pd.DataFrame({
    "mismo hospital, capital": compare(matched, "same_capital"),
    "mismo hospital, ponderado": compare(matched, "same_weighted"),
    "más cercano, capital (todos)": compare(comparison, "nearest_capital"),
    "más cercano, ponderado (todos)": compare(comparison, "nearest_weighted"),
}).T
display(summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, column, title in [(axes[0], "same_capital", "Desde la capital"),
                          (axes[1], "same_weighted", "Media ponderada por población")]:
    pair = matched[["miteco_minutes", column]].dropna()
    ax.scatter(pair[column], pair["miteco_minutes"], s=18)
    top = max(pair.max().max(), 1) * 1.05
    ax.plot([0, top], [0, top], color="grey", linewidth=1, label="iguales")
    ax.set_title(title)
    ax.set_xlabel("Nuestro tiempo (min)")
    ax.legend()
axes[0].set_ylabel("MITECO (min)")
fig.suptitle("Tiempo al mismo hospital por concejo")
plt.tight_layout()
plt.show()

## 4. Concejos con más diferencia

Con el mismo hospital. Diferencias grandes pueden venir de la red (carreteras de montaña a velocidad de llano), del
punto de origen (capital frente a otro punto) o de rutas que en la realidad pasan por fuera de Asturias.

In [ ]:
matched = matched.assign(diff_capital=matched["miteco_minutes"] - matched["same_capital"])
display(matched.reindex(matched["diff_capital"].abs().sort_values(ascending=False).index)
        [["municipality", "miteco_hospital", "miteco_minutes", "same_capital", "same_weighted", "diff_capital"]]
        .head(10).round(1))

## 5. Concejos donde el hospital no coincide

Para qué hospital da el MITECO el tiempo en los concejos que no hemos podido emparejar, y cuánto cambia el
tiempo por usar solo hospitales públicos de referencia.

In [ ]:
unmatched = comparison[comparison["service_id"].isna()]
display(unmatched.groupby("miteco_hospital").agg(
    concejos=("municipality", "size"),
    miteco_minutes=("miteco_minutes", "mean"),
    our_nearest_capital=("nearest_capital", "mean"),
).sort_values("concejos", ascending=False).round(1))

## Conclusiones (ejecución del 11/10/2026)

Con `asturias.gpkg` de la ejecución del 11/10 (red del extracto de Geofabrik del 08/10).

1. **Nuestros tiempos cuadran con los del MITECO.** En los 45 concejos con el mismo hospital, medidos desde la
   capital, el cociente mediano MITECO / nuestro es 0,97, el error absoluto medio es 1,6 min y la correlación de
   Spearman es 0,97. Ninguna diferencia pasa de 5,3 min (Pesoz).
2. **El MITECO parece medir desde la capital del concejo** (inferencia). La medida desde la capital se parece más
   a la suya (error de 1,6 min) que la media ponderada por población (2,6 min).
3. **Lo que esto valida y lo que no.** Valida la red, las velocidades y el enganche frente a otro modelo oficial
   hecho de forma parecida. No valida que los tiempos sean reales: los dos pueden quedarse cortos igual en montaña
   (los dos son tiempos de modelo, no medidos).
4. **El hospital sí cambia el resultado.** En los 33 concejos restantes el MITECO mide hasta otro hospital:
   privados (Centro Médico de Asturias en 14 concejos, Sanatorio Adaro en 6), psiquiátricos o de fuera de
   Asturias. Para preguntas sobre "hospital" hay que fijar la definición (la nuestra: los 9 públicos de
   referencia) y decirla en la respuesta.
5. **Hospitales de fuera.** En los concejos de borde que el MITECO manda a Sierrallana (Cantabria) o a Valle de
   Laciana (León), su tiempo es de 10 a 20 min menor que el nuestro a un hospital asturiano. Es el efecto de no
   tener hospitales ni carreteras de fuera de Asturias; afecta a 5 concejos y se declara como limitación.